# `07_requests.ipynb`

In [ ]:
# uv add requests
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'

res = requests.get(URL)


raw_data = res.text    # str -> Parsing 안된 데이터
data = res.json()  # dict -> Parsing 된 데이터 (해석됨, 활용 가능)

In [ ]:
# 1인당 1등 당첨 금액 = 'rnk1WnAmt'

data['data']['list'][0]['rnk1WnAmt']  # 1197258718

- 이번주 당첨 정보중 다음 데이터를 추출

```py
...
print(lucky)  # [2, 13, 18, 32, 38, 42]
print(bonus)  # 22
```

In [ ]:
# dict 도 for 로 순회가 가능하다!
d = {'a': 1, 'b': 2, 'c': 3}

for k, v in d.items():
    print(k, v)

In [ ]:
lucky = [1, 2, 3, 4, 5, 6]

my = [1, 2, 3, 4, 5, 6]


# 1: General
count = 0
for ball in lucky:
    if ball in my:
        count += 1

print(count)

# 2: Python 특화
len(set(lucky) & set(my))

In [ ]:
# Main Mission
# 랜덤하게 뽑은 번호 6개와, 실제 당첨번호를 비교하여
# 몇등인지 출력하는 프로그램. 완성하면
# (추가미션) 함수로 잘 만들기 -> 함수로 돌려서 1등 나올때까지 결과 기록

# 1등: 숫자 6개 같음
# 2등: 숫자 5개 같고 + 나머지 하나가 보너스번호
# 3등 ~ 5등: 숫자 5개, 4개, 3개 같음

# 랜덤번호 vs 실제 당첨번호
# -> 우선 고정번호 vs 실제 당첨번호

In [ ]:
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'
res = requests.get(URL)
data = res.json() 
core_data = data['data']['list'][0]

# 실제 당첨 숫자
lucky = []

for k, v in core_data.items():
    # 아! 모든 로또번호랑 연결된 key에는 'tm' 글자가 들어있군!
    if 'tm' in k:
        # key에 'tm' 들어간 경우에만 해당 value(로또번호)를 추가한다!
        lucky.append(v)
# 보너스 번호
bonus = core_data['bnsWnNo'] 

In [ ]:
import random

# 내가 랜덤하게 뽑은 숫자
my = random.sample(range(1, 46), 6)


match_count = len(set(lucky) & set(my))

if match_count == 6:
    result = '1'
elif match_count == 5 and bonus in my:
    result = '2'
elif match_count == 5:
    result = '3'
elif match_count == 4:
    result = '4'
elif match_count == 3:
    result = '5'
else:
    result = '꽝'

print(result)


In [ ]:
import requests

# 현실 로또 당첨 번호를 API 에서 가져옴
def fetch_lotto_info():
    URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'
    res = requests.get(URL)
    data = res.json() 
    core_data = data['data']['list'][0]

    lucky = []
    for k, v in core_data.items():
        if 'tm' in k:
            lucky.append(v)

    bonus = core_data['bnsWnNo']
    # 최종 return 값은 튜플 (1, 2)
    return lucky, bonus

In [ ]:
# 공주머니 2개랑 보너스를 넣으면 등수를 알려줌
def check_my_luck(my_nums, real_nums, bonus):
    match_count = len(set(my_nums) & set(real_nums))

    if match_count == 6:
        result = '1'
    elif match_count == 5 and bonus in my_nums:
        result = '2'
    elif match_count == 5:
        result = '3'
    elif match_count == 4:
        result = '4'
    elif match_count == 3:
        result = '5'
    else:
        result = '꽝'
    return result

In [ ]:
# 1. 정보 받기
lucky, bonus = fetch_lotto_info()

In [ ]:
import random

dashboard = {
    '1': 0, '2': 0, '3': 0,
    '4': 0, '5': 0, '꽝': 0,
}

# 대시보드에 1등 나온 횟수가 0번이면
while dashboard['1'] == 0:
    # 내번호 랜덤으로 뽑기
    my = random.sample(range(1, 46), 6)
    # 결과 비교하기
    result = check_my_luck(my, lucky, bonus)
    # 대시보드 기록
    dashboard[result] += 1

print(dashboard)


## API Key 관리
1. 터미널에 `uv add python-dotenv` 로 설치
2. 모든 키 파일은 `.env` 파일에 보관 (없으면 생성)
3. 소스코드에서는 `load_dotenv()` 와 `os.getenv()` 를 사용하여 불러옴

In [ ]:
import os
from dotenv import load_dotenv
import requests

# .env 파일 불러오기
load_dotenv()

# 불러온 파일에서 원하는 Key 꺼내기
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

In [ ]:
BASE_URL = 'https://naverapihub.apigw.ntruss.com'
NEWS_URL = '/search/v1/news'

# 인증 관련 헤더
headers = {
    'X-NCP-APIGW-API-KEY-ID': NAVER_CLIENT_ID,
    'X-NCP-APIGW-API-KEY': NAVER_CLIENT_SECRET
}

In [ ]:
URL = BASE_URL + NEWS_URL

# 쿼리 파라미터를 dict 로 작성
params = {
    'query': '엔화',
    'sort': 'sim',
    # 100개 기사를 모아서 (시작은 5개로)
    'display': 5,
}
res = requests.get(URL, headers=headers, params=params)

In [ ]:
# 100개 기사를 모아서
# title에 <b> </b> 이상한 태그 없애기 (검색 필요)
# 조건: link URL이 naver 뉴스인 애들만 모아야 함. 100개가 안될 수 있음.
# 간략히 다음과 같은 모양으로 만들기
# news 변수 내용을 csv 로 export 하기
data = res.json()['items']

news = []

for item in data:
    # 2. link에 naver가 없으면 버림
    if 'naver' in item['link']:
        # 1. <b> 없앤걸로 title 교체
        item['title'] = item['title'].replace('<b>', '').replace('</b>', '').replace('&quot;', '')  # 문자열에서 1번 인자를 2번 인자로 교체
        new_item = {
            'title': item['title'],
            'link': item['link']
        }
        news.append(new_item)

# for one_news in news:
#     requests.get(one_news['link'])

news

In [ ]:
import csv

filednames = news[0].keys()

with open('./news.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=filednames)
    writer.writeheader()
    writer.writerows(news)

## Parsing
1. JSON 문자열 -> dict 로 해석
2. HTML 문자열 -> 구조화 필요 (`BeautifulSoup4`)

In [ ]:
# uv add beautifulsoup4
import requests
from bs4 import BeautifulSoup


def extract_naver_news(url):
    # 네이버 뉴스 아니면 에러 발생
    if 'n.news.naver.com' not in url:
        raise Exception('네이버 뉴스가 아닙니다') 

    res = requests.get(url)
    # res.text 를 해석 완료!
    soup = BeautifulSoup(res.text, 'html.parser')
    # 해석한 HTML에서 '#dic_area' 선택자로 추출 -> 글자만 뽑아서 -> 양옆 공백(엔터, 스페이스) 삭제
    news_text = soup.select_one('#dic_area').text.strip()
    return news_text


URL = 'https://n.news.naver.com/article/008/0005405342'
extract_naver_news(URL)

In [ ]:
# 1. 특정 주제로 Naver News 연관도 순으로 5개 뽑기
# 2. Naver 뉴스 링크를 통해서 본문만 추출하기
# 3. 최종 결과는
'''
news = [
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
    {'title': '제목제목', 'link': '네이버뉴스링크', 'content': '추출한 본문'},
]
'''
# 4. 다 했으면, OpenAI 모듈 설치 후 AI로 댓글생성 해보기

## News 추출 총 정리
> 아래 코드만 확인하면 됨.

In [ ]:
import os
from dotenv import load_dotenv
import requests
from bs4 import BeautifulSoup

load_dotenv()

In [ ]:
def extract_naver_news(url):
    '''url 을 통해 naver news의 본문만 추출'''
    
    if 'n.news.naver.com' not in url:
        raise Exception('네이버 뉴스가 아닙니다') 

    res = requests.get(url)
    soup = BeautifulSoup(res.text, 'html.parser')
    news_text = soup.select_one('#dic_area').text.strip()
    return news_text

In [ ]:
def refine_news_list(naver_news_list: list[dict]):
    '''Naver 뉴스검색 API로 가져온 뉴스목록을 정제
        - title: 특수기호 삭제
        - link: 네이버뉴스 링크만
        - content: 뉴스 본문의 텍스트만 추출
    '''
    refined_list = []
    for item in naver_news_list:
        link = item['link']
        title = item['title']

        if 'naver' in item['link']:
            new_title = title.replace('<b>', '').replace('</b>', '').replace('&quot;', '')
            new_item = {
                'title': new_title,
                'link': link,
                'content': extract_naver_news(link)
            }
            refined_list.append(new_item)
    return refined_list

In [ ]:
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

# 세팅
BASE_URL = 'https://naverapihub.apigw.ntruss.com'
NEWS_URL = '/search/v1/news'
URL = BASE_URL + NEWS_URL

headers = {
    'X-NCP-APIGW-API-KEY-ID': NAVER_CLIENT_ID,
    'X-NCP-APIGW-API-KEY': NAVER_CLIENT_SECRET
}

params = {
    'query': '엔화',
    'sort': 'sim',
    'display': 5,
}


In [ ]:
res = requests.get(URL, headers=headers, params=params)
news_list = res.json()['items']

result = refine_news_list(news_list)
result

### OpenAI 사용하기
- Web API 방식 -> HTTP방식으로 OpenAI 서버에 요청을 보내서 응답을 받음
- SDK 방식 -> API를 더 쓰기 쉽게 만들어준 개발자 친화적 키트(kit)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# uv add openai
from openai import OpenAI

# OpenAI의 기능을 사용할 클라이언트 생성
client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY')
)

system_msg = '''
너는 매우 착한 댓글을 만들어주는 AI야. 
기사 내용을 보고 긍정적인 댓글을 만들어줘
'''
# 기사내용은 이미 추출해 놨음! 그걸 사용
user_msg = '루카 ART 수석 포트폴리오 매니저“시장, BOJ 추가 금리인상 과소평가”  美국채 비중 축소…“바이백은 미봉책”\n\n\n\n 일본 엔화. 로이터연합뉴스호주 2위 규모의 대형 연금기금인 ‘호주 연금 신탁(Australian Retirement Trust·ART)’이 일본은행(BOJ)의 기준금리 인상 가능성이 시장에 과소평가됐다고 판단하며 엔화 강세에 베팅하고 나섰다고 블룸버그통신이 26일(현지시간) 보도했다.약 3700억 호주달러(약 365조원)를 굴리는 ART의 지미 루카 수석 포트폴리오 매니저는 블룸버그와 인터뷰에서 “지난 6개월간 엔화 가치가 달러당 160엔에 육박하며 약세를 보이는 동안 미국 달러 비중을 일부 축소하고 엔화 포지션을 꾸준히 늘려왔다”고 밝혔다.이는 금융시장에서 엔화 약세를 예상하며 엔화 매도세가 이어지는 것과 정반대 흐름이어서 주목된다. 시장 참가자들은 에너지 대외 의존도가 높은 일본의 경제 구조에 주목, 계속되는 에너지 가격 상승과 BOJ의 미온적인 긴축 움직임 등을 바탕으로 엔화 약세에 베팅해왔다.루카 매니저는 ”시장이 고유가 악재는 이미 가격에 반영했지만, BOJ의 추가 금리 인상 가능성은 지나치게 낮게 보고 있다“며 ”현재 극도로 저평가된 엔화의 적정 환율은 달러당 150엔 수준이며, 140엔대 후반까지 강세를 보일 여력이 있다“고 진단했다. 현재 블룸버그가 집계한 스왑 시장 데이터에 따르면 9월 BOJ의 금리 인상 확률은 약 80%, 10월 인상은 100%다.한편 ART는 미국 자산에 대해서는 회의적인 시각을 유지하며 미 국채 비중을 축소(Underweight)했다. 목표치를 웃도는 인플레이션과 견조한 경제 성장, 인공지능(AI) 투자 붐에 따른 자본 수요 급증이 국채 금리를 계속 끌어올릴 것이라는 분석이다.루카 매니저는 스콧 베센트 미 재무장관이 최근 발표한 장기물 바이백 확대 조치를 단기 채권을 발행해 장기 채권을 사들이는 ‘오퍼레이션 트위스트(Operation Twist)’에 비유하며 “금리 상승 압력을 근본적으로 막기보다는 일시적으로 지연시키는 미봉책에 불과하다”고 평가했다. 이어 “미국 30년 만기 국채 금리가 연 5.5% 선을 향해 추가 상승 탄력을 받을 것”이라며 “명목 금리를 억누르려는 시도는 결국 인플레이션을 자극하고 달러화 가치를 떨어뜨릴 수 있다”고 경고했다.호주의 연기금 시장은 약 4조4000억 호주달러 규모로 세계에서 가장 큰 규모다. 연기금들은 전체 자산의 약 절반을 해외에 투자하고 있어 운용역들에게 환율 리스크 관리가 매우 중요하다. 다만 루카 매니저는 달러 자산은 여전히 중요한 분산투자 대상이라고 강조했다. 다만 현재 베선트 장관이 주도하는 장기 국채 금리의 의도적 인하 정책은 인플레이션을 자극하고 달러 가치를 떨어뜨릴 것이라는게 그의 진단이다.'

gpt_res = client.responses.create(
    model='gpt-4.1-mini',
    # system message
    instructions=system_msg,
    # user message
    input=user_msg
)

print(gpt_res.output_text)

In [ ]:
for item in result:
    gpt_res = client.responses.create(
      model='gpt-4.1-mini',
      # system message
      instructions='너는 매우 착한 댓글을 만들어주는 AI야. 기사 내용을 보고 긍정적인 댓글을 만들어줘',
      # user mesdsage
      input=item['content']
    )
    item['comment'] = gpt_res.output_text

In [ ]:
import csv

filednames = result[0].keys()

with open('./news.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=filednames)
    writer.writeheader()
    writer.writerows(result)

result

In [ ]:

s = ['http://naver.com\r', 'http://google.com\r', 'http://instagram.com\r', '']

list(map(lambda x: x.strip(), s))